In [1]:
# imports
import tensorflow as tf
import keras as keras
import matplotlib.pyplot as plt
import keras_tuner as kt

2024-05-20 23:52:21.927235: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-20 23:52:21.927364: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-20 23:52:21.929137: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-05-20 23:52:21.945750: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-20 23:52:23.212308: W tensorflow/compiler/tf2

In [2]:
IMG_SIZE = 224
BATCH_SIZE = 32
CHANNELS =3
input_shape = (IMG_SIZE, IMG_SIZE, CHANNELS)

def Model(input_shape):

    model = keras.Sequential([
        keras.layers.Conv2D(32,kernel_size=3,input_shape=input_shape),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Flatten(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(10, activation='softmax'),
    ])
    
    
    return model

Model(input_shape)
   

2024-05-20 23:52:26.423874: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-20 23:52:26.451151: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-20 23:52:26.451199: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-20 23:52:26.454752: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-20 23:52:26.454818: I external/local_xla/xla/stream_executor

In [3]:
#Hyperparameter tuning
model = Model(input_shape)
model.compile(optimizer = 'adam',
             loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits =False),
             metrics=['accuracy'])

In [4]:
#import  data

IMG_SIZE = 224
BATCH_SIZE = 32

train_data = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

test_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/test",
    shuffle = True,
    image_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE
)

val_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/valid",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

Found 2339 files belonging to 10 classes.


2024-05-20 23:52:39.099070: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Found 50 files belonging to 10 classes.
Found 50 files belonging to 10 classes.


In [5]:
data_augmentation_layers = [
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
]

def  data_augmentation(images):
    for layer in  data_augmentation_layers:
        images = layer(images)
    return images

def preprocess_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # Normalize pixel values to [0, 1]
    image = data_augmentation(image)
    return image, label

train_data = train_data.map(
    preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_data = val_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)


In [6]:

history = model.fit(
train_data,
epochs = 12,
batch_size = 32,
validation_data = val_data)

Epoch 1/12


2024-05-20 23:52:52.520634: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-05-20 23:52:53.108618: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-05-20 23:52:58.859621: I external/local_xla/xla/service/service.cc:168] XLA service 0x7fb0c030fd40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-05-20 23:52:58.859666: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2024-05-20 23:52:58.880160: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1716241978.985720  476815 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


74/74 [==============================] - 28s 138ms/step - loss: 2.0838 - accuracy: 0.2783 - val_loss: 4.0446 - val_accuracy: 0.1200
Epoch 2/12
74/74 [==============================] - 5s 68ms/step - loss: 1.6198 - accuracy: 0.3976 - val_loss: 5.0996 - val_accuracy: 0.1000
Epoch 3/12
74/74 [==============================] - 5s 66ms/step - loss: 1.4530 - accuracy: 0.4451 - val_loss: 5.1297 - val_accuracy: 0.0800
Epoch 4/12
74/74 [==============================] - 5s 68ms/step - loss: 1.3331 - accuracy: 0.4985 - val_loss: 3.6662 - val_accuracy: 0.1800
Epoch 5/12
74/74 [==============================] - 5s 65ms/step - loss: 1.2797 - accuracy: 0.5242 - val_loss: 2.1552 - val_accuracy: 0.3200
Epoch 6/12
74/74 [==============================] - 5s 66ms/step - loss: 1.1552 - accuracy: 0.5755 - val_loss: 2.5352 - val_accuracy: 0.1600
Epoch 7/12
74/74 [==============================] - 5s 65ms/step - loss: 1.0991 - accuracy: 0.5951 - val_loss: 2.7985 - val_accuracy: 0.3200
Epoch 8/12
74/74 [====

In [7]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

IMG_SIZE = 224
BATCH_SIZE = 32
CHANNELS = 3
input_shape = (IMG_SIZE, IMG_SIZE, CHANNELS)

def HModel(hp):
    model = keras.Sequential([
        keras.layers.Conv2D(32, kernel_size=3, input_shape=input_shape),
        keras.layers.MaxPool2D((2, 2)),
        
        keras.layers.Conv2D(32, kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.Conv2D(32, kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2, 2)),
        
        keras.layers.Conv2D(32, kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.Conv2D(32, kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2, 2)),
        
        keras.layers.Conv2D(32, kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.Conv2D(32, kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2, 2)),
        
        keras.layers.Conv2D(32, kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.Conv2D(32, kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2, 2)),
        
        keras.layers.Flatten(),
        keras.layers.Dropout(units = hp.Float("units", min_value = 0.1, max_value = 0.6, step =0.1)),
        keras.layers.Dense(units=hp.Int("units", min_value=32, max_value=512, step=32), activation="relu"),
        keras.layers.Dense(10, activation='softmax'),
    ])
    
    learning_rate = hp.Float("lr", min_value=1e-4, max_value=1e-2, sampling="log")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

HModel(kt.HyperParameters())

tuner = kt.RandomSearch(
    hypermodel=HModel,
    objective="val_accuracy",
    max_trials=3,
    executions_per_trial=2,
    overwrite=True,
    directory="./ACML_Project",
    project_name="ACML",
)
tuner.search_space_summary()

# Assuming train_data and val_data are properly prepared datasets
# For example, using tf.data.Dataset
# train_data = ...
# val_data = ...

tuner.search(train_data, epochs=2, validation_data=val_data)

models = tuner.get_best_models(num_models=2)
best_model = models[0]
best_model.summary()


TypeError: Dropout.__init__() missing 1 required positional argument: 'rate'

In [ ]:
#Retraining the model with the best hyperparamters
#Top 2
best = tuner.get_best_hyperparameters(5)

#Building model
model = HModel(best[0])

history = model.fit(
train_data,
epochs = 10,
batch_size = 32,
validation_data = val_data)